In [ ]:
pip install langchain langgraph openai faiss-cpu langchain_openai

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from typing import TypedDict
from langchain_core.tools import Tool
from langgraph.graph import StateGraph


In [ ]:
import os

# Replace 'YOUR_OPENAI_API_KEY' with your actual OpenAI API key
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"
llm = ChatOpenAI(
    temperature=0,
    model="gpt-4o-mini"  # or gpt-3.5
)

In [ ]:
# -------------------------
# 1. Tool
# -------------------------
def search_tool(query: str) -> str:
    query = query.lower()
    if "python" in query:
        return "Python interview topics: Variables, Data Types, OOP, Lists, Dictionaries, Functions"
    elif "sql" in query:
        return "SQL interview topics: SELECT, JOIN, WHERE, GROUP BY, Indexes, Normalization"
    elif "agent" in query or "genai" in query:
        return "GenAI interview topics: Agentic AI, RAG, Tools, Memory, ReAct"
    else:
        return f"General interview topics related to: {query}"

search = Tool(
    name="Search",
    func=search_tool,
    description="Find interview topics based on user goal"
)

In [ ]:
# -------------------------
# 2. State
# -------------------------
from typing import TypedDict

class AgentState(TypedDict):
  goal: str
  task: str
  research: str
  content: str
  review: str

In [ ]:
# -------------------------
# 3. Agents (Nodes = Functions)
# -------------------------
def manager_agent(state: AgentState)->AgentState:
    state["task"]=f"Prepare interview questions for {state['goal']}"
    return state
def research_agent(state: AgentState)->AgentState:
    state["research"]=search.invoke(state["goal"])
    return state


In [ ]:
def content_agent(state: AgentState) -> AgentState:
    prompt = f"""
You are a technical interviewer.

Generate 8 to 10 beginner-level interview questions with short, clear answers
for the following role or topic:

{state['goal']}

Use these reference topics:
{state['research']}

Format strictly as:
1. Question
   Answer:
"""
    response = llm.invoke(prompt)
    state["content"] = response.content
    return state


In [ ]:
def reviewer_agent(state: AgentState) -> AgentState:
    follow_up = (
        "\n\nFollow-up question:\n"
        "Would you like:\n"
        "1) More questions\n"
        "2) Topic-wise questions (Python / SQL / ML)\n"
        "3) Harder questions\n"
        "Type your choice."
    )
    state["review"] = state["content"] + follow_up
    return state

In [ ]:
graph=StateGraph(AgentState)
graph.add_node("manager",manager_agent)
graph.add_node("research",research_agent)
graph.add_node("content",content_agent)
graph.add_node("reviewer",reviewer_agent)

graph.set_entry_point("manager")
graph.add_edge("manager","research")
graph.add_edge("research","content")
graph.add_edge("content","reviewer")
app=graph.compile()

In [ ]:
user_goal=input("Enter your goal: ")
result =app.invoke({"goal": user_goal,
                    "task":"",
                    "research":"",
                    "content":"",
                    "review":""})
print("\n--------INTERIVEW QUESTIONS--------------------")
print(result["review"])
follow_up_input = input("\nEnter your choice (1 = More, 2 = Topic-wise, 3 = Harder): ")
print("\nYou selected:", follow_up_input)

In [ ]:
if follow_up_input == "3":
    hard_prompt = f"""
You are an interviewer.
Generate 8–10 HARDER interview questions with short answers
for the following role:

{user_goal}

Focus on:
- Conceptual depth
- Real-world scenarios
- Tricky questions

Return numbered Q&A.
"""
    hard_response = llm.invoke(hard_prompt)

    print("\nHARDER INTERVIEW QUESTIONS:\n")
    print(hard_response.content)

elif follow_up_input == "2":
    topic_prompt = f"""
Generate topic-wise interview questions with answers
for {user_goal}.

Sections:
- Python
- SQL
- Machine Learning
"""
    topic_response = llm.invoke(topic_prompt)

    print("\nTOPIC-WISE INTERVIEW QUESTIONS:\n")
    print(topic_response.content)

elif follow_up_input == "1":
    more_prompt = f"""
Generate 8 more beginner-level interview questions
with short answers for {user_goal}.
"""
    more_response = llm.invoke(more_prompt)

    print("\nMORE INTERVIEW QUESTIONS:\n")
    print(more_response.content)

else:
    print("\nInvalid choice. Please select 1, 2, or 3.")
